In [ ]:
%pip -q install "numpy>=2.1,<3" "pandas>=2.2,<3" "rasterio>=1.4,<2" matplotlib
from google.colab import drive, files
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import sys
import json

uploaded = files.upload()
required = ['ndvi_pipeline.py', 'ndvi_candidate.py', 'canberra_ndvi.geojson']
missing = set(required) - set(uploaded)
if missing:
    raise ValueError(f'Missing uploads: {sorted(missing)}')
for name in required:
    Path('/content', name).write_bytes(uploaded[name])
sys.path.insert(0, '/content')
importlib.invalidate_caches()
for name in ['ndvi_candidate', 'ndvi_pipeline']:
    sys.modules.pop(name, None)
import ndvi_candidate as candidate

PROJECT_DIR = Path('/content/drive/MyDrive/SAT_SCOPE')
RAW_DIR = PROJECT_DIR / 'data/raw/mod13q1'
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = PROJECT_DIR / 'validation/candidate_runs' / run_id
print('Candidate version:', candidate.VERSION)
print('Separate output directory:', OUTPUT_DIR)


In [ ]:
result = candidate.run_candidate(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    geometry_path=Path('/content/canberra_ndvi.geojson'),
    baseline_start=2020,
    baseline_end=2024,
    evaluation_year=2025,
    minimum_years=4,
    minimum_coverage=80.0,
)
print(json.dumps(result, indent=2))
assert result['deployment_allowed'] is False


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

evaluation = pd.read_csv(OUTPUT_DIR / 'candidate_evaluation.csv')
display(evaluation[['date', 'current', 'baseline', 'change_percent',
                    'matched_coverage_percent', 'minimum_pixel_history', 'status']])
manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
print('Missing periods:', manifest['missing_periods'])
print('Validation status:', manifest['validation_status'])

dates = pd.to_datetime(evaluation['date'])
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(dates, evaluation['current'], 'o-', label='Matched current median')
axes[0].plot(dates, evaluation['baseline'], 'o--', label='Median of pixel seasonal baselines')
axes[0].set_ylabel('NDVI')
axes[0].legend()
axes[1].plot(dates, evaluation['matched_coverage_percent'], 'o-')
axes[1].axhline(80, linestyle='--', color='gray', label='Provisional reporting rule')
axes[1].set_ylabel('Matched coverage (%)')
axes[1].legend()
fig.suptitle('Candidate validation only; gaps indicate unavailable reportable values')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'candidate_time_series.png', dpi=150)
plt.show()


In [ ]:
# Compare with your earlier diagnostic export if it exists.
prior_path = PROJECT_DIR / 'validation/matched_pixel_change_sensitivity.csv'
if prior_path.exists():
    prior = pd.read_csv(prior_path)
    prior = prior[prior['required_years'] == 4]
    comparison = evaluation.merge(prior[['date', 'change_percent']], on='date',
                                  suffixes=('_candidate', '_earlier'), validate='one_to_one')
    comparison['difference_pp'] = (comparison['diagnostic_change_percent']
                                  - comparison['change_percent_earlier'])
    display(comparison[['date', 'diagnostic_change_percent',
                        'change_percent_earlier', 'difference_pp', 'status']])
    discrepancy = comparison['difference_pp'].abs().max()
    print('Largest difference from earlier diagnostic (percentage points):', discrepancy)
    print('This is a reproducibility comparison, not independent validation.')
else:
    print('Earlier diagnostic CSV not found. Compare against the previously saved tables.')


In [ ]:
import zipfile
archive = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}_candidate.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            z.write(path, arcname=path.name)
files.download(str(archive))
print('For review only. Do not copy candidate files to the website data directory.')
